# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook explores the FAIR² dataset on mass spectrometry analysis of extracellular vesicles isolated from stimulated human mast cells using the `mlcroissant` library for FAIR dataset loading and EDA.

### Dataset Source
This analysis uses the Croissant schema available here:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata object (not as dict or list, per instructions)
meta = dataset.metadata
print(f"Dataset title: {meta.name}\nDescription: {meta.description}")

## 2. Data Overview
Review available record sets and fields. All entities are referenced by their `@id` attributes.

Let's print all record sets and the fields they contain.

In [ ]:
# List all record sets by @id
meta_json = meta.to_json()  # safe to extract the structure for pretty printing

if 'recordSet' in meta_json and meta_json['recordSet']:
    record_sets = meta_json['recordSet']
    print(f"Available record sets: {len(record_sets)}\n")
    for record_set in record_sets:
        rs_id = record_set['@id']
        rs_name = record_set.get('name', rs_id)
        print(f"Record set: {rs_id} ({rs_name})")
        # List fields by @id in this recordset
        if 'field' in record_set:
            print("  Fields:")
            for field in record_set['field']:
                print(f"    - {field['@id']}: {field.get('name', field['@id'])}")
        print()
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from a record set into a DataFrame for analysis. Use record set and field `@id` attributes.

Below, we extract all available record sets by their `@id` and load them into pandas DataFrames. Adjust the IDs in the following cell based on the overview above.

In [ ]:
# Extract data from each record set
meta_json = meta.to_json()
# Build list of record set @id attributes
if 'recordSet' in meta_json and meta_json['recordSet']:
    record_sets = [rs['@id'] for rs in meta_json['recordSet']]
else:
    record_sets = []

dataframes = {}
# Iterate by @id and print columns
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set {record_set_id} with {df.shape[0]} rows and columns: {df.columns.tolist()}")
        print(df.head(), '\n')
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Select one for further analysis, e.g. the first available
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"Selecting record set for EDA: {selected_record_set_id}")
    print(f"Columns: {dataframes[selected_record_set_id].columns.tolist()}")
    dataframes[selected_record_set_id].head()
else:
    selected_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps (filtering, normalizing, grouping). We select a numeric field by its `@id` for operations below.

In [ ]:
import numpy as np

if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]
    print(f"Working with DataFrame from record set: {selected_record_set_id}")
    # Try to select a numeric column for demonstration
    sample_numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if sample_numeric_candidates:
        numeric_field_id = sample_numeric_candidates[0]
        print(f"Using numeric field for EDA: {numeric_field_id}")
        threshold = df[numeric_field_id].mean()  # Example threshold at mean
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered to records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} rows")
        
        # Normalize (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Try categorical/grouping
        group_candidates = df.select_dtypes(include=["object", "category"]).columns.tolist()
        if group_candidates:
            group_field_id = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            group_field_id = None
            print("No categorical columns found to group by.")
    else:
        numeric_field_id = None
        group_field_id = None
        print("No numeric fields found in the selected record set.")
else:
    numeric_field_id = None
    group_field_id = None
    print("No data available for EDA.")

## 5. Visualization
Let's visualize the distribution of a numeric field and, if available, its group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field_id is not None:
        plt.figure(figsize=(9, 5))
        order = df[group_field_id].value_counts().index
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, order=order)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² mass spectrometry dataset using the `mlcroissant` library. We reviewed accessible record sets and fields (always referencing them by their `@id`), extracted data for analysis, performed basic EDA including filtering and normalization, and visualized key numeric distributions. This workflow demonstrates a reproducible path for further downstream analysis or machine learning using FAIR Croissant datasets.